# LFM Semantic Segmentation Inference
This notebook is an example workflow of doing semantic segmentation on visible light, UV, and static bands of Lunar data, using the 512x512 pixel "datacubes", generated by the Pipeline workflow. 

## Purpose of this notebook
This notebook is designed to be used after semantic segmentation training has been run (see semantic_seg_train.ipynb in the repo). Once you have run that, a checkpoint model will be saved to your disk (should be saved under {path/to/your/repo}/lfm/notebooks/outputs/sem_seg/checkpoints by default). Find the path to the best model checkpoint (usually called "best_model.pt"), and put the full path value in the config section below (variable is called "pretrained_checkpoint"). Once you have this, configure the other variables as you desire (comments in the cell can guide you), and run the rest of the notebook!

**Note**: currently, the pretrained model this notebook uses was trained on 12-band input data (5 VIS bands, 2 UV bands, 5 Kaguya static bands). If you wish to train your own model with a different input dataset (i.e. different static band configuration), that will be added soon. If you would like to filter out certain WAC/Static bands, you can filter them using the BAND_FILTER variable in the config. **See the README in the [LFM repo](https://github.com/nasa-nccs-hpda/lfm) for more info.** 

## Getting started

### Downloading
You can get started with this notebook by downloading it with:

```bash
wget https://raw.githubusercontent.com/nasa-nccs-hpda/lfm/refs/heads/main/notebooks/cube_inference_sseg.ipynb
```

### Code dependencies -- terminal command
The notebook requires some code from the [LFM repo](https://github.com/nasa-nccs-hpda/lfm/tree/main). To install this code, navigate to the top of the JupyterHub user interface, and click on the box with the + symbol. Then scroll down, and choose the "Terminal" (under Other section, first option). 

Then, if it's your first time using any of the LFM notebooks, run:

```bash
cd path/to/lfm/notebooks
git clone https://github.com/nasa-nccs-hpda/lfm.git
```

Or, if you've already run the above command, run:

```bash
cd path/to/lfm/notebooks
git pull https://github.com/nasa-nccs-hpda/lfm/tree/main
```

This will get you the most up-to-date code to support the notebook. 

### Python installs
You will notice 2 folders below that install dependencies for you; one removes your "local python installs", and the other installs things locally. **This is completely harmless, and don't worry about anything important getting deleted**. Local installs are very easy to get back using pip or conda, we are just deleting previous installs here to make sure we have a clean environment to work with. 

**See the README in the [repo](https://github.com/nasa-nccs-hpda/lfm)** for more info on how to use this notebook, and more on the process of training the model. 

# Setup

In [ ]:
!rm -rf ~/.local/lib/python*

In [ ]:
!pip install xarray tiler rioxarray transformers torchmetrics # For H100 system

In [ ]:
# Standard library imports
import os
import subprocess
import sys
from datetime import datetime
from glob import glob
from pathlib import Path

# Third-party imports
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import numpy.ma as ma
import torch
import xarray as xr
from tqdm import tqdm
from tiler import Tiler, Merger

%matplotlib inline

In [ ]:
repo_dir = "lfm"
sys.path.insert(0, os.path.join(os.getcwd(), repo_dir))

from lfm.tasks.sem_segmentation.sseg_model import load_dinov3_encoder, DINOSegmentation
from lfm.tasks.sem_segmentation.data_cube_inference import run_datacube_inference
from lfm.tasks.sem_segmentation.sseg_dataset import get_input_metadata

# User Configuration
`INPUT_ROOT_DIR`: where the tiling outputs went 

`OUTPUT_DIR`: this is a relative path, so by default the outputs (visualizations, model checkpoints, dataset statistics) will go to the same folder as the notebook. 

`pretrained_checkpoint`: where to load the pretrained semantic segmentation model from; **if you haven't run that notebook yet, get the notebook with**: 

```bash
wget https://github.com/nasa-nccs-hpda/lfm/blob/develop/notebooks/semantic_seg.ipynb
```

`BAND_FILTER`: list of band indices (0-indexed) to use in the dataset; for example, if I want to use only VIS bands, I would supply [0, 1, 2, 3, 4]. Band ordering is the following for the 12-band dataset: VIS (0-4), UV (5, 6), KAGUYA STATIC (7-11). 

In [ ]:
# Data paths
INPUT_ROOT_DIR = "/explore/nobackup/projects/lfm/tiling_outputs/WAC_Processed_AOI"

# Where to load dinov3 init weights, and where to load the trained model weights
pretrained_checkpoint = "/explore/nobackup/projects/lfm/model_inference/checkpoints/sem_seg/12_band/checkpoint_epoch_100_NO_ZSCORE.pt"

# Output dir (this will be created automatically)
OUTPUT_DIR = Path("./outputs/inference")  # Change this if you want a specific path
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
print(f"Output directory: {OUTPUT_DIR}")

# Number of bands, band filter
BAND_FILTER = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]  # Bands to keep, in order of filtering

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load model

In [ ]:
# Get input metadata based on band filter
weight_assignments = get_input_metadata(INPUT_ROOT_DIR, BAND_FILTER)

In [ ]:
# Load model from model factory
encoder = load_dinov3_encoder(device=device)
model = DINOSegmentation(
    encoder=encoder,  # Use DINOv3 head
    num_classes=2,  # Binary segmentaton (crater, not crater)
    weight_assignments=weight_assignments
).to(device)

# Apply checkpoint
checkpoint = torch.load(pretrained_checkpoint, map_location='cpu')
checkpoint_state = checkpoint['model_state_dict']
missing_keys, unexpected_keys = model.load_state_dict(
    checkpoint_state, strict=False
)
model.to(device)
model.eval()
print("Successfully loaded model from checkpoint!")

# Inference

In [ ]:
images, preds = run_datacube_inference(
    model=model,
    device=device,
    input_dir=INPUT_ROOT_DIR,
    output_dir=OUTPUT_DIR,
    band_filter=BAND_FILTER
)